In [1]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, accuracy_score


In [14]:
import pandas as pd
import re

# --------------------------------------------------
# 1️⃣ Load RAW CSV safely (very important)
# --------------------------------------------------
raw_df = pd.read_csv(
    "tamil_audio_dataset.csv",
    encoding="utf-8",
    sep=",",
    quotechar='"',
    engine="python"
)

print("Raw shape:", raw_df.shape)
print("Raw columns:", raw_df.columns.tolist())

# Force correct column names
raw_df.columns = ["text", "label"]

# --------------------------------------------------
# 2️⃣ Inspect raw label distribution (DEBUG STEP)
# --------------------------------------------------
print("\nRaw label distribution:")
print(raw_df["label"].value_counts(dropna=False))

# --------------------------------------------------
# 3️⃣ Clean TEXT column (STRONG but SAFE)
# --------------------------------------------------
def clean_tamil_text(text):
    text = str(text)

    # Remove quotes (straight & curly)
    text = text.replace('"', '')
    text = text.replace('“', '').replace('”', '')

    # Remove trailing punctuation only
    text = re.sub(r'[.,!?]+$', '', text)

    # Keep Tamil Unicode block + space
    text = re.sub(r'[^\u0B80-\u0BFF\s]', ' ', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

raw_df["clean_text"] = raw_df["text"].apply(clean_tamil_text)

# --------------------------------------------------
# 4️⃣ Clean LABEL column VERY CAREFULLY
# --------------------------------------------------
raw_df["label"] = (
    raw_df["label"]
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
    .str.lower()
)

# IMPORTANT: do NOT drop anything yet
print("\nLabels after string cleaning:")
print(raw_df["label"].value_counts(dropna=False))

# --------------------------------------------------
# 5️⃣ Map labels → numeric (SAFE mapping)
# --------------------------------------------------
label_map = {
    "dep": 1,
    "depressed": 1,

    "ndep": 0,
}

raw_df["label_num"] = raw_df["label"].map(label_map)

# --------------------------------------------------
# 6️⃣ Check mapping result BEFORE dropping
# --------------------------------------------------
print("\nAfter label mapping (before drop):")
print(raw_df["label_num"].value_counts(dropna=False))

# --------------------------------------------------
# 7️⃣ Drop ONLY truly invalid rows
# --------------------------------------------------
clean_df = raw_df.dropna(subset=["label_num", "clean_text"])

# Remove empty text rows
clean_df = clean_df[clean_df["clean_text"].str.strip() != ""]

clean_df["label"] = clean_df["label_num"].astype(int)

# --------------------------------------------------
# 8️⃣ Final sanity checks
# --------------------------------------------------
print("\nFINAL CLEANED DATASET")
print("Shape:", clean_df.shape)
print("Label distribution:")
print(clean_df["label"].value_counts())

# --------------------------------------------------
# 9️⃣ Keep only required columns
# --------------------------------------------------
final_df = clean_df[["clean_text", "label"]]

# --------------------------------------------------
# 🔟 Save cleaned CSV
# --------------------------------------------------
final_df.to_csv(
    "tamil_text_cleaned.csv",
    index=False,
    encoding="utf-8"
)

print("\n✅ tamil_text_cleaned.csv saved successfully!")
final_df.head()


Raw shape: (1375, 2)
Raw columns: ['audio_text', 'label']

Raw label distribution:
label
ndep                919
dep                 454
ndepwhich ffmpeg      1
NaN                   1
Name: count, dtype: int64

Labels after string cleaning:
label
ndep                919
dep                 454
ndepwhich ffmpeg      1
NaN                   1
Name: count, dtype: int64

After label mapping (before drop):
label_num
0.0    919
1.0    454
NaN      2
Name: count, dtype: int64

FINAL CLEANED DATASET
Shape: (1364, 4)
Label distribution:
label
0    911
1    453
Name: count, dtype: int64

✅ tamil_text_cleaned.csv saved successfully!


,clean_text,label
0,ஒருவாட்டி கூட நீங்கள் யாருமே என்னைப் புரிந்துக...,1
1,நான் இப்போது வாழ்க்கும் வாழ்க்கையும் என்னுக்கு...,1
2,எனக்கு இப்ப தான் தோன்றுது நான் உன்னை எவ்வளவு ந...,1
3,கடவைத் திறந்திருக்கா திறக்க முடியுமா என்னால் இ...,1
4,ஏதோ பெருஸ்வா நிலைக்கப்போகும் மாதிரி எனக்கு மிக...,1
